<a href="https://colab.research.google.com/github/dasmiq/cs6200-documents/blob/main/PageRank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PageRank

In this exercise, you will compute PageRank on a collection of 469,235 web sites using the iterative version of the PageRank algorithm described in class for sparse graphs.  As we discussed in class, this avoids performing cubic-time operations on a graph with a large number of nodes.

Furthermore, you should take advantage of the _sparsity_ of the original hyperlinks, even though "teleportation" or random jumps connect every note to every other node.  As long as the maximum degree of the original link graph is much less than the number of nodes, you should be able to keep each iteration's runtime less than quadratic in the number of nodes.

Consider the following directed graph:

![A directed link graph](https://www.khoury.northeastern.edu/home/dasmith/courses/cs6200//pagerank.jpg)

We can represent this graph as a collection of nodes, here, ordered pairs of node index and node name:

In [1]:
small_nodes = [(0, 'A'),
              (1, 'B'),
              (2, 'C'),
              (3, 'D'),
              (4, 'E'),
              (5, 'F')]

and a collection of directed links, i.e., ordered pairs from source to target:

In [2]:
small_edges = [
  (0, 1),
  (0, 2),
  (0, 5),
  (1, 2),
  (1, 3),
  (1, 4),
  (1, 5),
  (2, 3),
  (2, 4),
  (3, 0),
  (3, 2),
  (3, 4),
  (3, 5),
  (4, 0),
  (5, 0),
  (5, 1),
  (5, 4)
]

We use integer identifiers for the nodes for efficiency. In most real-world collections of hyperlinks, unlike this example, not every page will have in-links, nor will every page have out-links.

## First Implementation and Test

\[10 points\] Implement the iterative PageRank algorithm we discussed in class. Test your code on the six-node example using the input representation given above.  Be sure that your code handles pages that have no in-links or out-links properly.  (You may wish to test on a few such examples.) In later parts of this assignment, depending on how you store the data, it may be convenient to use iterators rather than storing the data in memory.

In [6]:
# TODO: Implement PageRank, given nodes and edges, to start with a uniform
# distribution over nodes, run a fixed number of iterations, and
# return a distribution over nodes.


def page_rank_fixed_iter(nodes, edges, iterations=10, damping=0.85):
    """
    Computes PageRank using sparse graph iterations.
    
    Parameters:
    - nodes: list of tuples (node_id, node_name)
    - edges: list of directed edges (source_id, target_id)
    - iterations: number of iterations to run
    - damping: damping factor lambda (default 0.85)
    
    Returns:
    - list of tuples: (node_id, node_name, score)
    """
    N = len(nodes)
    if N == 0:
        return []

    # Map node index to its name
    node_names = {idx: name for idx, name in nodes}
    all_indices = list(node_names.keys())

    # 1. Precompute out-degrees and in-links
    out_degree = {idx: 0 for idx in all_indices}
    in_links = {idx: [] for idx in all_indices}

    for src, dst in edges:
        out_degree[src] += 1
        in_links[dst].append(src)

    # 2. Identify sink nodes (nodes with 0 out-links)
    sink_nodes = [idx for idx in all_indices if out_degree[idx] == 0]

    # 3. Initialize uniform probability distribution
    pr = {idx: 1.0 / N for idx in all_indices}

    # 4. Power iteration loop
    for _ in range(iterations):
        # Accumulate probability mass from dangling/sink nodes
        sink_mass = sum(pr[idx] for idx in sink_nodes)
        
        # Teleport mass shared uniformly by all nodes
        base_score = (1.0 - damping + damping * sink_mass) / N
        
        # Compute new scores using sparse in-links
        new_pr = {}
        for idx in all_indices:
            link_score = sum(pr[src] / out_degree[src] for src in in_links[idx])
            new_pr[idx] = base_score + damping * link_score
            
        pr = new_pr

    # 5. Format output with (node_number, name, PageRank value)
    return [(idx, node_names[idx], pr[idx]) for idx in all_indices]

# Output PageRank on the toy graph at various points.
# Make sure your output has node number, name, and PageRank value.
# Convert the list to a string first

print(page_rank_fixed_iter(small_nodes, small_edges, 1), end="\n\n")
print(page_rank_fixed_iter(small_nodes, small_edges, 10), end="\n\n")
print(page_rank_fixed_iter(small_nodes, small_edges, 100), end="\n\n")

[(0, 'A', 0.24930555555555556), (1, 'B', 0.11944444444444445), (2, 'C', 0.14305555555555555), (3, 'D', 0.13125), (4, 'E', 0.21388888888888888), (5, 'F', 0.14305555555555555)]

[(0, 'A', 0.25203637602817186), (1, 'B', 0.13930650918251078), (2, 'C', 0.15129376593475072), (3, 'D', 0.1189629727768988), (4, 'E', 0.1871066101429171), (5, 'F', 0.15129376593475072)]

[(0, 'A', 0.2521271053751955), (1, 'B', 0.13930618531853856), (2, 'C', 0.15130648986670525), (3, 'D', 0.11890782257353918), (4, 'E', 0.18704590699931606), (5, 'F', 0.15130648986670525)]



## PageRank on Web Crawl Data

\[20 points\] Download and unpack a list of `.edu` websites and the links among them from the [Common Crawl](https://commoncrawl.org/2017/05/hostgraph-2017-feb-mar-apr-crawls/) open-source web crawl. For the sake of brevity, the data record links among websites, not web pages. The information for nodes and links is the same as the toy example above.

In [7]:
# If you're running on a machine (e.g., Windows) that doesn't have wget or gzip,
# feel free to comment this out and use a different set of commands to load
# the data.
!wget https://khoury.northeastern.edu/home/dasmith/courses/cs6200/vertices-edu.txt.gz
!gzip -df vertices-edu.txt.gz
!wget https://khoury.northeastern.edu/home/dasmith/courses/cs6200/edges-edu.txt.gz
!gzip -df edges-edu.txt.gz

--2026-10-07 18:51:26--  https://khoury.northeastern.edu/home/dasmith/courses/cs6200/vertices-edu.txt.gz
Resolving khoury.northeastern.edu (khoury.northeastern.edu)... 52.70.229.197
Connecting to khoury.northeastern.edu (khoury.northeastern.edu)|52.70.229.197|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3703486 (3.5M) [application/x-gzip]
Saving to: ‘vertices-edu.txt.gz’

vertices-edu.txt.gz 100%[===================>]   3.53M  16.2MB/s    in 0.2s    

2026-10-07 18:51:26 (16.2 MB/s) - ‘vertices-edu.txt.gz’ saved [3703486/3703486]

--2026-10-07 18:51:27--  https://khoury.northeastern.edu/home/dasmith/courses/cs6200/edges-edu.txt.gz
Resolving khoury.northeastern.edu (khoury.northeastern.edu)... 52.70.229.197
Connecting to khoury.northeastern.edu (khoury.northeastern.edu)|52.70.229.197|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12829526 (12M) [application/x-gzip]
Saving to: ‘edges-edu.txt.gz’

edges-edu.txt.gz    100%[===========

There should now be files `vertices-edu.txt` and `edges-edu.txt`.

In [8]:
# TODO: Process the raw data into the same format as the simple graph.
# You may create lists or iterators.

# 1. Load vertices / nodes
edu_nodes = []
with open("vertices-edu.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            parts = line.split()
            node_id = int(parts[0])
            node_name = parts[1]
            edu_nodes.append((node_id, node_name))

# 2. Load directed edges
edu_edges = []
with open("edges-edu.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            parts = line.split()
            src = int(parts[0])
            dst = int(parts[1])
            edu_edges.append((src, dst))

print(f"Total nodes loaded: {len(edu_nodes):,}")
print(f"Total edges loaded: {len(edu_edges):,}")

Total nodes loaded: 469,235
Total edges loaded: 3,300,462


Refine your implementation of PageRank to test for numerical convergence. Specificially, at each iteration, calculate the [perplexity](https://en.wikipedia.org/wiki/Perplexity) of the PageRank distribution, where perplexity is defined as 2 raised to the [Shannon entropy](https://en.wikipedia.org/wiki/Entropy_(information_theory)) of the PageRank distribution, i.e., $2^{H(PR)}$. (Recall the definition of entropy from our discussion of data compression.) The maximum perplexity of a PageRank distribution will therefore be the number of nodes in the graph.

At each iteration, check the _change_ in perplexity. If the change is less than some threshold, you can stop.


In [9]:
# TODO: Implement convergence testing in PageRank
# If you choose, you can share some subroutines with your first version.
# Print the change in perplexity at each iteration.


import math

def calculate_perplexity(pr_scores):
    """
    Computes 2^(Shannon Entropy) for a probability distribution.
    H(PR) = -sum(p * log2(p))
    """
    entropy = 0.0
    for p in pr_scores:
        if p > 0:
            entropy -= p * math.log2(p)
    return 2.0 ** entropy


def page_rank(nodes, edges, threshold=1, damping=0.85):
    """
    Computes PageRank using sparse iterations until the change in perplexity
    is strictly less than the given threshold.
    """
    N = len(nodes)
    if N == 0:
        return []

    node_names = [name for _, name in nodes]

    # 1. Precompute out-degrees and in-links
    out_degree = [0] * N
    in_links = [[] for _ in range(N)]

    for src, dst in edges:
        out_degree[src] += 1
        in_links[dst].append(src)

    # Pre-identify dangling/sink nodes (out-degree == 0)
    sink_nodes = [i for i in range(N) if out_degree[i] == 0]

    # Pre-scale inverse out-degrees
    inv_out_degree = [1.0 / deg if deg > 0 else 0.0 for deg in out_degree]

    # 2. Initial uniform distribution
    pr = [1.0 / N] * N
    prev_perplexity = calculate_perplexity(pr)
    print(f"Initial Perplexity (Iteration 0): {prev_perplexity:,.4f}")

    iteration = 0
    while True:
        iteration += 1

        # Collect dangling mass
        sink_mass = sum(pr[i] for i in sink_nodes)
        base_score = (1.0 - damping + damping * sink_mass) / N

        # Sparse power iteration step
        new_pr = [0.0] * N
        for dst in range(N):
            link_score = sum(pr[src] * inv_out_degree[src] for src in in_links[dst])
            new_pr[dst] = base_score + damping * link_score

        pr = new_pr

        # 3. Compute new perplexity and change
        curr_perplexity = calculate_perplexity(pr)
        diff = abs(curr_perplexity - prev_perplexity)

        print(f"Iteration {iteration}: Perplexity = {curr_perplexity:,.4f}, Change = {diff:,.4f}")

        # Convergence test
        if diff < threshold:
            print(f"\nConverged after {iteration} iterations (Change < {threshold}).")
            break

        prev_perplexity = curr_perplexity

    return [(i, node_names[i], pr[i]) for i in range(N)]


# Run until perplexity changes by less than 1
PR = page_rank(edu_nodes, edu_edges, 1)

Initial Perplexity (Iteration 0): 469,235.0000
Iteration 1: Perplexity = 298,071.4462, Change = 171,163.5538
Iteration 2: Perplexity = 255,742.8147, Change = 42,328.6314
Iteration 3: Perplexity = 234,252.2722, Change = 21,490.5426
Iteration 4: Perplexity = 225,975.9559, Change = 8,276.3163
Iteration 5: Perplexity = 221,538.9998, Change = 4,436.9561
Iteration 6: Perplexity = 219,533.9646, Change = 2,005.0352
Iteration 7: Perplexity = 218,300.9225, Change = 1,233.0421
Iteration 8: Perplexity = 217,671.9390, Change = 628.9835
Iteration 9: Perplexity = 217,238.0146, Change = 433.9244
Iteration 10: Perplexity = 216,999.3836, Change = 238.6310
Iteration 11: Perplexity = 216,815.2333, Change = 184.1504
Iteration 12: Perplexity = 216,708.5297, Change = 106.7036
Iteration 13: Perplexity = 216,619.0678, Change = 89.4619
Iteration 14: Perplexity = 216,564.5220, Change = 54.5458
Iteration 15: Perplexity = 216,516.2282, Change = 48.2937
Iteration 16: Perplexity = 216,485.3513, Change = 30.8769
Iter

## Link Analysis

\[20 points\] In this final section, you will compute some properties of this web-site graph.

First, consider the _in-link count_ of a website, simply the number of websites pointing to it (including self-links).

In [10]:
# TODO: List the document ID, domain name, and in-link count of the 70 websites with the highest in-link count

from collections import Counter

# 1. Count incoming edges for every destination node in edu_edges
in_link_counts = Counter()
for src, dst in edu_edges:
    in_link_counts[dst] += 1

# 2. Map integer node ID to domain name string
name_lookup = {node_id: name for node_id, name in edu_nodes}

# 3. Sort all nodes by in-link count descending
sorted_by_inlinks = sorted(
    edu_nodes, 
    key=lambda node: in_link_counts[node[0]], 
    reverse=True
)

# 4. Extract top 70 websites
top_70_inlinks = sorted_by_inlinks[:70]

# 5. Display the results formatted as Document ID, Domain Name, and In-link count
print(f"{'Rank':<6}{'Doc ID':<10}{'In-Links':<12}{'Domain Name'}")
print("-" * 55)
for rank, (node_id, name) in enumerate(top_70_inlinks, start=1):
    count = in_link_counts[node_id]
    print(f"{rank:<6}{node_id:<10}{count:<12}{name}")

Rank  Doc ID    In-Links    Domain Name
-------------------------------------------------------
1     185524    4388        edu.mit.web
2     278032    4021        edu.stanford
3     244433    3531        edu.purdue.english.owl
4     140443    3339        edu.indiana
5     237176    3251        edu.princeton
6     64587     3123        edu.columbia
7     465503    2804        edu.yale
8     418623    2622        edu.utexas
9     383763    2592        edu.unc
10    197698    2494        edu.nap
11    439637    2291        edu.washington
12    373442    2281        edu.umich
13    440674    2276        edu.washington.depts
14    148945    2255        edu.jhu.muse
15    60975     2232        edu.colorado
16    449738    2230        edu.wisc
17    38320     2205        edu.bu
18    83572     1965        edu.dartmouth
19    408380    1952        edu.usc
20    178879    1946        edu.mit
21    27307     1908        edu.berkeley
22    233405    1857        edu.pitt
23    191069    1810     

Then, use the PageRank values compute by your second implementation. Note that some websites will have both a high in-link count and PageRank.

In [11]:
# TODO: List the document ID, domain name, and PageRank of the 70 websites with the highest PageRank.

# 1. Sort the converged PageRank results (PR) in descending order of score
# Each element in PR is (node_id, domain_name, score)
top_70_pagerank = sorted(PR, key=lambda item: item[2], reverse=True)[:70]

# 2. Display the Document ID, Domain Name, and PageRank score
print(f"{'Rank':<6}{'Doc ID':<10}{'PageRank':<16}{'Domain Name'}")
print("-" * 60)
for rank, (doc_id, domain_name, score) in enumerate(top_70_pagerank, start=1):
    print(f"{rank:<6}{doc_id:<10}{score:<16.8f}{domain_name}")

Rank  Doc ID    PageRank        Domain Name
------------------------------------------------------------
1     185524    0.00092800      edu.mit.web
2     465503    0.00070059      edu.yale
3     318278    0.00064157      edu.ucdavis.cas
4     278032    0.00059447      edu.stanford
5     136464    0.00057970      edu.illinois
6     319795    0.00054653      edu.ucdavis.ice.code
7     237176    0.00050196      edu.princeton
8     284517    0.00047478      edu.stanford.web
9     383763    0.00047168      edu.unc
10    64587     0.00044737      edu.columbia
11    449738    0.00043621      edu.wisc
12    60975     0.00043240      edu.colorado
13    346969    0.00043005      edu.ucsf
14    140443    0.00042962      edu.indiana
15    42502     0.00041783      edu.byu.cas
16    334739    0.00041279      edu.uconn
17    14945     0.00040928      edu.arizona
18    244433    0.00040303      edu.purdue.english.owl
19    373442    0.00040208      edu.umich
20    439637    0.00039843      edu.washi

Finally, compute some summary statistics on this dataset.

In [ ]:
# TODO: Compute:
# - the proportion of websites with no in-links (i.e., source nodes);
# - the proportion of websites with no out-links (i.e., sink nodes);
# - the proportion of websites whose PageRank is higher than the initial uniform distribution.


N = len(edu_nodes)
initial_uniform_pr = 1.0 / N

# 1. Track in-degrees and out-degrees across all nodes
in_degree = [0] * N
out_degree = [0] * N

for src, dst in edu_edges:
    out_degree[src] += 1
    in_degree[dst] += 1

# 2. Count source nodes (no in-links) and sink nodes (no out-links)
no_in_links = sum(1 for d in in_degree if d == 0)
no_out_links = sum(1 for d in out_degree if d == 0)

# 3. Count nodes whose converged PageRank exceeds the initial uniform value
higher_than_uniform = sum(1 for _, _, score in PR if score > initial_uniform_pr)

# 4. Compute proportions
prop_no_in = no_in_links / N
prop_no_out = no_out_links / N
prop_higher_pr = higher_than_uniform / N

# 5. Display the results
print(f"Total websites (N): {N:,}\n")

print(f"1. Proportion with no in-links (sources): {prop_no_in:.4f} ({prop_no_in * 100:.2f}%) [{no_in_links:,} nodes]")
print(f"2. Proportion with no out-links (sinks):  {prop_no_out:.4f} ({prop_no_out * 100:.2f}%) [{no_out_links:,} nodes]")
print(f"3. Proportion with PageRank > 1/N:        {prop_higher_pr:.4f} ({prop_higher_pr * 100:.2f}%) [{higher_than_uniform:,} nodes]")

Total websites (N): 469,235

1. Proportion with no in-links (sources): 0.2615 (26.15%) [122,722 nodes]
2. Proportion with no out-links (sinks):  0.6107 (61.07%) [286,545 nodes]
3. Proportion with PageRank > 1/N:        0.1551 (15.51%) [72,800 nodes]
